# COVID-19 Forecasting with Snowflake ML
Train two `SNOWFLAKE.ML.FORECAST` models to predict `ROLLING_7DAY_CASES` and `ROLLING_7DAY_DEATHS`, one model per country.

In [ ]:
%%sql -r setup_result
USE WAREHOUSE SYSTEM$STREAMLIT_NOTEBOOK_WH;
USE DATABASE HACKATHON;
USE SCHEMA SG01;

In [ ]:
%%sql -r view_cases
CREATE OR REPLACE VIEW HACKATHON.SG01.V_TRAIN_CASES AS
SELECT DATE::TIMESTAMP_NTZ AS DATE, COUNTRY_REGION, ROLLING_7DAY_CASES::FLOAT AS ROLLING_7DAY_CASES
FROM HACKATHON.SG01.COVID_DEATH_FEATURES
WHERE ROLLING_7DAY_CASES IS NOT NULL;

In [ ]:
%%sql -r view_deaths
CREATE OR REPLACE VIEW HACKATHON.SG01.V_TRAIN_DEATHS AS
SELECT DATE::TIMESTAMP_NTZ AS DATE, COUNTRY_REGION, ROLLING_7DAY_DEATHS::FLOAT AS ROLLING_7DAY_DEATHS
FROM HACKATHON.SG01.COVID_DEATH_FEATURES
WHERE ROLLING_7DAY_DEATHS IS NOT NULL;

In [ ]:
%%sql -r model_cases
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST HACKATHON.SG01.covid_cases_forecast(
  INPUT_DATA => TABLE(HACKATHON.SG01.V_TRAIN_CASES),
  SERIES_COLNAME => 'COUNTRY_REGION',
  TIMESTAMP_COLNAME => 'DATE',
  TARGET_COLNAME => 'ROLLING_7DAY_CASES'
);

In [ ]:
%%sql -r model_deaths
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST HACKATHON.SG01.covid_deaths_forecast(
  INPUT_DATA => TABLE(HACKATHON.SG01.V_TRAIN_DEATHS),
  SERIES_COLNAME => 'COUNTRY_REGION',
  TIMESTAMP_COLNAME => 'DATE',
  TARGET_COLNAME => 'ROLLING_7DAY_DEATHS'
);

In [ ]:
%%sql -r eval_cases
CALL HACKATHON.SG01.covid_cases_forecast!SHOW_EVALUATION_METRICS();

In [ ]:
%%sql -r eval_deaths
CALL HACKATHON.SG01.covid_deaths_forecast!SHOW_EVALUATION_METRICS()

In [ ]:
%%sql -r forecast_cases
CALL HACKATHON.SG01.covid_cases_forecast!FORECAST(FORECASTING_PERIODS => 30);

In [ ]:
%%sql -r forecast_deaths
CALL HACKATHON.SG01.covid_deaths_forecast!FORECAST(FORECASTING_PERIODS => 30);

## Store Forecast Predictions
Save the 30-day forecast results into tables for downstream use.

In [ ]:
%%sql -r store_cases
CREATE OR REPLACE TABLE HACKATHON.SG01.FORECAST_ROLLING_7DAY_CASES AS
SELECT
    SERIES::VARCHAR AS COUNTRY_REGION,
    TS AS FORECAST_DATE,
    FORECAST AS PREDICTED_ROLLING_7DAY_CASES,
    LOWER_BOUND,
    UPPER_BOUND
FROM TABLE(HACKATHON.SG01.covid_cases_forecast!FORECAST(FORECASTING_PERIODS => 30));

In [ ]:
%%sql -r store_deaths
CREATE OR REPLACE TABLE HACKATHON.SG01.FORECAST_ROLLING_7DAY_DEATHS AS
SELECT
    SERIES::VARCHAR AS COUNTRY_REGION,
    TS AS FORECAST_DATE,
    FORECAST AS PREDICTED_ROLLING_7DAY_DEATHS,
    LOWER_BOUND,
    UPPER_BOUND
FROM TABLE(HACKATHON.SG01.covid_deaths_forecast!FORECAST(FORECASTING_PERIODS => 30));

In [ ]:
%%sql -r preview_cases
SELECT * FROM HACKATHON.SG01.FORECAST_ROLLING_7DAY_CASES LIMIT 10;

In [ ]:
%%sql -r preview_deaths
SELECT * FROM HACKATHON.SG01.FORECAST_ROLLING_7DAY_DEATHS LIMIT 10;

## COVID Vaccinations Forecasting
Train a `SNOWFLAKE.ML.FORECAST` model to predict `NEW_VACCINATIONS`, one model per country.

In [ ]:
%%sql -r view_vax
CREATE OR REPLACE VIEW HACKATHON.SG01.V_TRAIN_VACCINATIONS AS
SELECT DATE::TIMESTAMP_NTZ AS DATE, COUNTRY_REGION, NEW_VACCINATIONS::FLOAT AS NEW_VACCINATIONS
FROM HACKATHON.SG01.COVID_VACCINES_FEATURES
WHERE NEW_VACCINATIONS IS NOT NULL;

In [ ]:
%%sql -r model_vax
CREATE OR REPLACE SNOWFLAKE.ML.FORECAST HACKATHON.SG01.covid_vaccinations_forecast(
  INPUT_DATA => TABLE(HACKATHON.SG01.V_TRAIN_VACCINATIONS),
  SERIES_COLNAME => 'COUNTRY_REGION',
  TIMESTAMP_COLNAME => 'DATE',
  TARGET_COLNAME => 'NEW_VACCINATIONS'
);

In [ ]:
%%sql -r eval_vax
CALL HACKATHON.SG01.covid_vaccinations_forecast!SHOW_EVALUATION_METRICS();

In [ ]:
%%sql -r forecast_vax
CALL HACKATHON.SG01.covid_vaccinations_forecast!FORECAST(FORECASTING_PERIODS => 30);

In [ ]:
%%sql -r save_vax_forecast
CREATE OR REPLACE TABLE HACKATHON.SG01.COVID_VACCINATIONS_FORECAST_RESULTS AS
SELECT *
FROM TABLE(HACKATHON.SG01.covid_vaccinations_forecast!FORECAST(FORECASTING_PERIODS => 30));